# Hierarchical Bayesian Flight Fare Prediction

This notebook follows the style of the professor's example: short sections, one idea per cell, and clear explanations before each code block.

The goal is to model flight fares with a hierarchical Bayesian model in Pyro, using category-specific random intercepts and a linear effect for numerical features.

## 1. Imports

We first load the libraries used for data processing, Bayesian inference, and evaluation.

In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pyro
import pyro.distributions as dist
import pyro.optim as optim
import torch
from pyro.infer import MCMC, NUTS, Predictive, SVI, Trace_ELBO
from pyro.infer.autoguide import AutoNormal
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split

## 2. Load the dataset

The data is loaded from the Excel file in the project folder. We keep only the columns used in the final model and convert the target from INR to USD, as in the team baseline.

In [3]:
data_path = Path('Cleaned_dataset-2.xlsx')
df = pd.read_excel(data_path)

selected_cols = [
    'Journey_day',
    'Airline',
    'Source',
    'Departure',
    'Total_stops',
    'Arrival',
    'Destination',
    'Duration_in_hours',
    'Days_left',
    'Fare',
]
df = df[selected_cols].dropna().copy()
df['Fare'] = (df['Fare'].astype(float) * 0.01066).round(0)

print('Dataset shape:', df.shape)
df.head()

Dataset shape: (452088, 10)


,Journey_day,Airline,Source,Departure,Total_stops,Arrival,Destination,Duration_in_hours,Days_left,Fare
0,Monday,SpiceJet,Delhi,After 6 PM,non-stop,After 6 PM,Mumbai,2.0833,1,57.0
1,Monday,Indigo,Delhi,After 6 PM,non-stop,Before 6 AM,Mumbai,2.3333,1,63.0
2,Monday,GO FIRST,Delhi,After 6 PM,non-stop,Before 6 AM,Mumbai,2.1667,1,62.0
3,Monday,SpiceJet,Delhi,After 6 PM,non-stop,After 6 PM,Mumbai,2.0833,1,62.0
4,Monday,Air India,Delhi,After 6 PM,non-stop,After 6 PM,Mumbai,2.1667,1,63.0


## 3. Train/test split

We split the data before fitting any preprocessing step. This avoids leakage and keeps the evaluation honest.

In [4]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

categorical_cols = ['Journey_day', 'Airline', 'Source', 'Departure', 'Total_stops', 'Arrival', 'Destination']
numeric_cols = ['Duration_in_hours', 'Days_left']

print('Train size:', len(train_df))
print('Test size:', len(test_df))

Train size: 361670
Test size: 90418


## 4. Helper functions

We fit categorical encodings and standardization parameters on the training set only. Unknown categories in the test set are mapped to a dedicated placeholder index.

In [5]:
def encode_train_test(train_series, test_series):
    categories = sorted(train_series.astype(str).unique().tolist())
    mapping = {cat: i for i, cat in enumerate(categories)}
    unk_idx = len(mapping)
    mapping['__UNK__'] = unk_idx

    train_encoded = train_series.astype(str).map(mapping).fillna(unk_idx).astype(int).to_numpy()
    test_encoded = test_series.astype(str).map(mapping).fillna(unk_idx).astype(int).to_numpy()
    return train_encoded, test_encoded, len(mapping), mapping

def zscore_train_test(train_col, test_col):
    mean = float(train_col.mean())
    std = float(train_col.std())
    if std == 0:
        std = 1.0
    train_scaled = ((train_col - mean) / std).to_numpy(dtype=np.float32)
    test_scaled = ((test_col - mean) / std).to_numpy(dtype=np.float32)
    return train_scaled, test_scaled, mean, std

## 5. Encode features

Categorical features become integer indices, numerical features are standardized, and the target is scaled in the same way.

## 6.1. The generative process

Before writing the Pyro model, it is useful to describe the data-generating process in words. The observed predictors are treated as given inputs, while the fare is generated from a hierarchical linear model.

A simple generative story is:

1. Draw a global intercept $\beta_0$.
2. For each categorical feature, draw a group-level scale $\sigma_{cat,i}$ and then one random intercept for each category level:
   $\alpha_{cat,i} \sim \mathcal{N}(0, \sigma_{cat,i})$.
3. Draw regression coefficients for the numerical predictors:
   $\beta_{num} \sim \mathcal{N}(0, I)$.
4. Combine all effects into the linear predictor:
   $\mu = \beta_0 + \sum_i \alpha_{cat,i}[x_i] + x_{num}^T \beta_{num}$.
5. Finally, generate the observed fare:
   $Fare \sim \mathcal{N}(\mu, \sigma_{obs})$.

This structure matches the hierarchy we want: each category can have its own baseline effect, but those effects are still shared through the common prior. This is exactly what makes the model partially pooled and more stable than fitting each category independently.

In [6]:
cat_train_tensors = []
cat_test_tensors = []
cardinalities = []
category_mappings = {}

for col in categorical_cols:
    tr, te, card, mapping = encode_train_test(train_df[col], test_df[col])
    cat_train_tensors.append(torch.tensor(tr, dtype=torch.long))
    cat_test_tensors.append(torch.tensor(te, dtype=torch.long))
    cardinalities.append(card)
    category_mappings[col] = mapping

x_train_num = []
x_test_num = []
numeric_scalers = {}
for col in numeric_cols:
    tr, te, mean, std = zscore_train_test(train_df[col].astype(float), test_df[col].astype(float))
    x_train_num.append(tr)
    x_test_num.append(te)
    numeric_scalers[col] = {'mean': mean, 'std': std}

x_train = torch.tensor(np.stack(x_train_num, axis=1), dtype=torch.float32)
x_test = torch.tensor(np.stack(x_test_num, axis=1), dtype=torch.float32)

y_train_scaled, y_test_scaled, fare_mean, fare_std = zscore_train_test(
    train_df['Fare'].astype(float),
    test_df['Fare'].astype(float),
)
y_train = torch.tensor(y_train_scaled, dtype=torch.float32)
y_test = torch.tensor(y_test_scaled, dtype=torch.float32)

print('Cardinalities:', cardinalities)
print('Target mean/std:', fare_mean, fare_std)

Cardinalities: [8, 10, 8, 5, 4, 5, 8]
Target mean/std: 243.5633229186828 216.46799082474848


## 6. Hierarchical Pyro model

The model has a global intercept, category-specific random intercepts, a linear effect for the numerical predictors, and a Gaussian observation model for the fare.

In [7]:
def flight_model(cat_tensors, num_tensor, cardinalities, fare=None):
    n_obs = num_tensor.shape[0]

    global_intercept = pyro.sample('global_intercept', dist.Normal(0.0, 1.0))

    mu_categorical = 0.0
    for i, cat_tensor in enumerate(cat_tensors):
        sigma_cat = pyro.sample(f'sigma_cat_{i}', dist.HalfCauchy(1.0))
        with pyro.plate(f'plate_cat_{i}', cardinalities[i]):
            alpha_cat = pyro.sample(f'alpha_cat_{i}', dist.Normal(0.0, sigma_cat))
        mu_categorical = mu_categorical + alpha_cat[cat_tensor]

    beta_num = pyro.sample(
        'beta_num',
        dist.Normal(torch.zeros(num_tensor.shape[1]), torch.ones(num_tensor.shape[1])).to_event(1),
    )
    mu_numerical = (num_tensor * beta_num).sum(-1)

    sigma_obs = pyro.sample('sigma_obs', dist.HalfCauchy(1.0))
    mu = global_intercept + mu_categorical + mu_numerical

    with pyro.plate('data', n_obs):
        pyro.sample('obs', dist.Normal(mu, sigma_obs), obs=fare)

## 7. Evaluation helper

This helper computes point-prediction metrics and the 90% predictive interval coverage.

In [8]:
def metrics_from_predictive_samples(pred_samples, y_test, fare_mean, fare_std):
    pred_median_scaled = torch.median(pred_samples, dim=0).values
    pred_lower_scaled = torch.quantile(pred_samples, 0.05, dim=0)
    pred_upper_scaled = torch.quantile(pred_samples, 0.95, dim=0)

    mae_scaled = torch.mean(torch.abs(pred_median_scaled - y_test)).item()
    rmse_scaled = torch.sqrt(torch.mean((pred_median_scaled - y_test) ** 2)).item()
    coverage_90 = ((y_test >= pred_lower_scaled) & (y_test <= pred_upper_scaled)).float().mean().item()

    pred_median_usd = pred_median_scaled.detach().cpu().numpy() * fare_std + fare_mean
    y_test_usd = y_test.detach().cpu().numpy() * fare_std + fare_mean

    mae_usd = mean_absolute_error(y_test_usd, pred_median_usd)
    rmse_usd = np.sqrt(mean_squared_error(y_test_usd, pred_median_usd))

    return {
        'mae_scaled': float(mae_scaled),
        'rmse_scaled': float(rmse_scaled),
        'coverage_90': float(coverage_90),
        'mae_usd': float(mae_usd),
        'rmse_usd': float(rmse_usd),
    }

## 8. Variational inference training

We train the model with SVI and the AutoNormal guide. This is the fast approximate posterior used as the main working solution.

In [9]:
pyro.clear_param_store()
guide = AutoNormal(flight_model)
optimizer = optim.Adam({'lr': 0.03})
svi = SVI(model=flight_model, guide=guide, optim=optimizer, loss=Trace_ELBO())

losses = []
for epoch in range(500):
    loss = svi.step(cat_train_tensors, x_train, cardinalities, fare=y_train)
    loss_per_sample = loss / len(y_train)
    losses.append(loss_per_sample)
    if epoch % 100 == 0:
        print(f'Epoch {epoch:4d} | ELBO loss per sample: {loss_per_sample:.4f}')

print('Training complete')

Epoch    0 | ELBO loss per sample: 1.4710
Epoch  100 | ELBO loss per sample: 1.3225
Epoch  200 | ELBO loss per sample: 1.3041
Epoch  300 | ELBO loss per sample: 1.3048
Epoch  400 | ELBO loss per sample: 1.3020
Training complete


## 9. VI predictions and metrics

We sample from the approximate posterior and evaluate the test-set predictions.

In [10]:
predictive = Predictive(
    model=flight_model,
    guide=guide,
    num_samples=500,
    return_sites=('obs',),
)
samples = predictive(cat_test_tensors, x_test, cardinalities, fare=None)
pred_samples = samples['obs']

vi_metrics = metrics_from_predictive_samples(pred_samples, y_test, fare_mean, fare_std)
print(json.dumps(vi_metrics, indent=2))

{
  "mae_scaled": 0.7121599316596985,
  "rmse_scaled": 0.8886858820915222,
  "coverage_90": 0.9274702072143555,
  "mae_usd": 154.15982055664062,
  "rmse_usd": 192.37204554261515
}


## 10. Optional MCMC comparison

The professor recommended trying different inference algorithms. Here we keep an optional NUTS comparison on a smaller subset because it is more expensive than VI.

In [11]:
run_mcmc = False

if run_mcmc:
    mcmc_size = min(2000, len(y_train))
    mcmc_idx = torch.randperm(len(y_train))[:mcmc_size]
    cat_train_mcmc = [c[mcmc_idx] for c in cat_train_tensors]
    x_train_mcmc = x_train[mcmc_idx]
    y_train_mcmc = y_train[mcmc_idx]

    nuts = NUTS(flight_model, target_accept_prob=0.85)
    mcmc = MCMC(nuts, num_samples=200, warmup_steps=200, num_chains=1)
    mcmc.run(cat_train_mcmc, x_train_mcmc, cardinalities, fare=y_train_mcmc)

    posterior_samples = mcmc.get_samples()
    predictive_mcmc = Predictive(flight_model, posterior_samples=posterior_samples, return_sites=('obs',))
    samples_mcmc = predictive_mcmc(cat_test_tensors, x_test, cardinalities, fare=None)
    mcmc_metrics = metrics_from_predictive_samples(samples_mcmc['obs'], y_test, fare_mean, fare_std)
    print(json.dumps(mcmc_metrics, indent=2))
else:
    print('MCMC comparison is disabled in this notebook cell.')

MCMC comparison is disabled in this notebook cell.


## 11. Synthetic recovery check

This cell is a validation step: we generate artificial data from the prior and check whether the inference procedure can approximately recover the latent parameters. This is a strong implementation sanity check.

In [12]:
def run_parameter_recovery_check(cat_tensors_train, x_train, cardinalities, epochs=100, lr=0.03):
    device = x_train.device
    n_obs = x_train.shape[0]
    n_num = x_train.shape[1]

    true_intercept = torch.randn((), device=device)
    true_beta_num = torch.randn(n_num, device=device)
    true_sigma_obs = torch.distributions.HalfCauchy(torch.tensor(1.0, device=device)).sample()

    mu_categorical = torch.zeros(n_obs, device=device)
    for i, cat_tensor in enumerate(cat_tensors_train):
        sigma_cat = torch.distributions.HalfCauchy(torch.tensor(1.0, device=device)).sample()
        alpha = torch.randn(cardinalities[i], device=device) * sigma_cat
        mu_categorical = mu_categorical + alpha[cat_tensor]

    mu = true_intercept + mu_categorical + (x_train * true_beta_num).sum(-1)
    synthetic_fare = torch.normal(mu, true_sigma_obs)

    pyro.clear_param_store()
    recovery_guide = AutoNormal(flight_model)
    recovery_svi = SVI(model=flight_model, guide=recovery_guide, optim=optim.Adam({'lr': lr}), loss=Trace_ELBO())
    for _ in range(epochs):
        recovery_svi.step(cat_tensors_train, x_train, cardinalities, fare=synthetic_fare)

    recovered = recovery_guide.median()
    recovered_intercept = recovered['global_intercept'].detach().cpu().item()
    recovered_beta_num = recovered['beta_num'].detach().cpu().numpy()

    return {
        'intercept_abs_error': float(abs(recovered_intercept - true_intercept.detach().cpu().item())),
        'beta_num_mae': float(np.mean(np.abs(recovered_beta_num - true_beta_num.detach().cpu().numpy()))),
        'true_sigma_obs': float(true_sigma_obs.detach().cpu().item()),
    }

recovery_metrics = run_parameter_recovery_check(cat_train_tensors, x_train, cardinalities, epochs=100)
print(json.dumps(recovery_metrics, indent=2))

{
  "intercept_abs_error": 0.2848623991012573,
  "beta_num_mae": 0.06065613031387329,
  "true_sigma_obs": 1.7868123054504395
}


## 12. Save a short summary

This final cell stores the most important outputs so they can be reused outside the notebook.

In [ ]:
summary = {
    'vi_metrics': vi_metrics,
    'recovery_metrics': recovery_metrics,
    'categorical_cols': categorical_cols,
    'numeric_cols': numeric_cols,
}

Path('notebook_summary.json').write_text(json.dumps(summary, indent=2))
print('Notebook summary saved to notebook_summary.json')
print(json.dumps(summary, indent=2))

Notebook summary saved to notebook_summary.json
